# 04 — Client Pitch Analysis

**Purpose:** Generate all the insights needed for a client pitch deck.  
**How to use:** Change the `CLIENT` and `CATEGORY` variables in cell 3, or leave as None to auto-pick the top ones.  
**Output:** Spend share, benchmarks, demographics, trends, and talking points.

**Tables used:** `analytics.int_customer_category_spend`, `analytics.int_destination_metrics`, `marts.mart_monthly_trends`, `marts.mart_demographic_summary`, `marts.mart_destination_benchmarks`, `marts.mart_geo_summary`

In [ ]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import os
PROJECT = os.environ.get('BQ_PROJECT', 'fmn-sandbox')
LOCATION = 'africa-south1'
client = bigquery.Client(project=PROJECT, location=LOCATION)

def q(sql):
    return client.query(sql).to_dataframe()

print(f'Connected to {PROJECT} ({LOCATION})')

## Configuration
**Change these two variables for each pitch. Run all cells below.**

In [ ]:
# change these if you want a specific client, otherwise it picks the top one automatically
CLIENT = None                       # set to eg 'Adidas' or leave None for auto
CATEGORY = None                     # set to eg 'Clothing & Apparel' or leave None for auto
TOP_N_COMPETITORS = 5

# auto-pick the biggest category and top client if not set
if not CATEGORY:
    top_cat = q(f"""
        SELECT CATEGORY_TWO, SUM(total_spend) AS spend
        FROM `{PROJECT}.marts.mart_destination_benchmarks`
        GROUP BY CATEGORY_TWO ORDER BY spend DESC LIMIT 1
    """)
    CATEGORY = top_cat.iloc[0]['CATEGORY_TWO']

if not CLIENT:
    top_client = q(f"""
        SELECT DESTINATION, total_spend
        FROM `{PROJECT}.marts.mart_destination_benchmarks`
        WHERE CATEGORY_TWO = '{CATEGORY}'
        ORDER BY total_spend DESC LIMIT 1
    """)
    CLIENT = top_client.iloc[0]['DESTINATION']

print(f'Pitch target: {CLIENT}')
print(f'Category: {CATEGORY}')
print(f'Competitors: top {TOP_N_COMPETITORS}')
print(f'\n(change CLIENT and CATEGORY above to pick a diferent one)')

---
## 1. Client headline KPIs
The numbers that go on slide 1 of the pitch deck.

In [ ]:
kpi = q(f"""
    SELECT 
        CAST(customers AS INT64) AS customers,
        CAST(transactions AS INT64) AS transactions,
        ROUND(total_spend, 0) AS total_spend,
        ROUND(avg_txn_value, 2) AS avg_txn_value,
        ROUND(spend_per_customer, 0) AS spend_per_customer,
        ROUND(market_share_pct, 2) AS market_share_pct,
        ROUND(penetration_pct, 2) AS penetration_pct,
        CAST(spend_rank AS INT64) AS spend_rank
    FROM `{PROJECT}.marts.mart_destination_benchmarks`
    WHERE CATEGORY_TWO = '{CATEGORY}' AND DESTINATION = '{CLIENT}'
""")

if not kpi.empty:
    k = kpi.iloc[0]
    print(f'--- {CLIENT} in {CATEGORY} ---')
    print(f'  Customers:        {k["customers"]:,}')
    print(f'  Total spend:      R{k["total_spend"]:,.0f}')
    print(f'  Market share:     {k["market_share_pct"]}%')
    print(f'  Penetration:      {k["penetration_pct"]}%')
    print(f'  Avg transaction:  R{k["avg_txn_value"]:,.2f}')
    print(f'  Spend/customer:   R{k["spend_per_customer"]:,.0f}')
    print(f'  Rank in category: #{k["spend_rank"]}')
else:
    print(f'No data found for {CLIENT} in {CATEGORY}.')

## 2. Share of wallet
What % of each customer's category spend goes to the client?

In [ ]:
sow = q(f"""
    WITH client_customers AS (
        SELECT UNIQUE_ID, SUM(dest_spend) AS client_spend, AVG(share_of_wallet_pct) AS sow
        FROM `{PROJECT}.analytics.int_customer_category_spend`
        WHERE CATEGORY_TWO = '{CATEGORY}' AND DESTINATION = '{CLIENT}'
        GROUP BY UNIQUE_ID
    ),
    cat_customers AS (
        SELECT COUNT(DISTINCT UNIQUE_ID) AS total_cat_customers
        FROM `{PROJECT}.analytics.int_customer_category_spend`
        WHERE CATEGORY_TWO = '{CATEGORY}'
    )
    SELECT
        (SELECT total_cat_customers FROM cat_customers) AS category_customers,
        COUNT(*) AS client_customers,
        ROUND(SAFE_DIVIDE(COUNT(*) * 100.0, (SELECT total_cat_customers FROM cat_customers)), 1) AS penetration,
        ROUND(AVG(sow), 1) AS avg_share_of_wallet,
        ROUND(SUM(client_spend), 0) AS total_client_spend
    FROM client_customers
""")

if not sow.empty and sow.iloc[0]['client_customers'] > 0:
    s = sow.iloc[0]
    print(f'{CLIENT} captures {s["avg_share_of_wallet"]}% of their customers\' {CATEGORY} spend on average.')
    print(f'Penetration: {int(s["client_customers"]):,} out of {int(s["category_customers"]):,} category customers ({s["penetration"]}%)')
else:
    print(f'No share of wallet data for {CLIENT} in {CATEGORY}.')

In [ ]:
bands = q(f"""
    SELECT
        CASE
            WHEN share_of_wallet_pct >= 80 THEN '80-100% (Loyalist)'
            WHEN share_of_wallet_pct >= 50 THEN '50-80% (Primary)'
            WHEN share_of_wallet_pct >= 20 THEN '20-50% (Secondary)'
            ELSE '1-20% (Occasional)'
        END AS wallet_band,
        COUNT(DISTINCT UNIQUE_ID) AS customers,
        ROUND(SUM(dest_spend), 0) AS spend
    FROM `{PROJECT}.analytics.int_customer_category_spend`
    WHERE CATEGORY_TWO = '{CATEGORY}' AND DESTINATION = '{CLIENT}'
    GROUP BY wallet_band ORDER BY wallet_band
""")

if not bands.empty:
    fig = px.bar(bands, x='wallet_band', y='customers', color='spend',
                 color_continuous_scale='Greens',
                 title=f'{CLIENT} -- share of wallet distribution')
    fig.show()
else:
    print(f'No wallet data for {CLIENT} in {CATEGORY}.')

## 3. Competitive benchmarks
Client vs top competitors with real names. Anonymization happens in the final dashboard only.

In [ ]:
bench = q(f"""
    SELECT * FROM `{PROJECT}.marts.mart_destination_benchmarks`
    WHERE CATEGORY_TWO = '{CATEGORY}'
    ORDER BY total_spend DESC
""")

if not bench.empty:
    client_row = bench[bench['DESTINATION'] == CLIENT].copy()
    competitors = bench[bench['DESTINATION'] != CLIENT].head(TOP_N_COMPETITORS).copy()

    if client_row.empty:
        print(f'{CLIENT} not found in {CATEGORY} benchmarks.')
    else:
        display = pd.concat([client_row, competitors]).sort_values('total_spend', ascending=False)
        colors = [('#2E75B6' if d == CLIENT else '#607D8B') for d in display['DESTINATION']]

        fig = go.Figure()
        fig.add_trace(go.Bar(x=display['DESTINATION'], y=display['market_share_pct'],
                             marker_color=colors))
        fig.update_layout(title=f'{CLIENT} vs top {TOP_N_COMPETITORS} competitors -- market share (%)',
                          yaxis_title='Market share %')
        fig.show()

        display[['DESTINATION', 'customers', 'total_spend', 'avg_txn_value',
                 'spend_per_customer', 'market_share_pct', 'penetration_pct']]
else:
    print(f'No benchmark data for {CATEGORY}.')

In [ ]:
if 'display' in dir() and not display.empty:
    colors = [('#2E75B6' if d == CLIENT else '#607D8B') for d in display['DESTINATION']]
    fig = go.Figure()
    fig.add_trace(go.Bar(x=display['DESTINATION'], y=display['spend_per_customer'],
                         marker_color=colors))
    fig.update_layout(title=f'Spend per customer (R) -- {CLIENT} vs competitors',
                      yaxis_title='R per customer')
    fig.show()

    if not client_row.empty and not competitors.empty:
        cr = client_row.iloc[0]
        avg_comp = competitors['market_share_pct'].mean()
        if cr['market_share_pct'] > avg_comp:
            print(f'\n{CLIENT} LEADS with {cr["market_share_pct"]}% market share vs competitor avg of {avg_comp:.1f}%')
        else:
            print(f'\n{CLIENT} TRAILS at {cr["market_share_pct"]}% market share vs competitor avg of {avg_comp:.1f}% -- growth opportunity')
else:
    print('No benchmark data available.')

## 4. Monthly trends
Client spend vs category total over time.

In [ ]:
cat_trends = q(f"""
    SELECT month, SUM(total_spend) AS category_spend
    FROM `{PROJECT}.marts.mart_monthly_trends`
    WHERE CATEGORY_TWO = '{CATEGORY}'
    GROUP BY month ORDER BY month
""")

client_trends = q(f"""
    SELECT month, SUM(total_spend) AS client_spend
    FROM `{PROJECT}.marts.mart_monthly_trends`
    WHERE CATEGORY_TWO = '{CATEGORY}' AND DESTINATION = '{CLIENT}'
    GROUP BY month ORDER BY month
""")

if not cat_trends.empty and not client_trends.empty:
    trends = cat_trends.merge(client_trends, on='month', how='left').fillna(0)
    trends['client_share_pct'] = round(
        trends['client_spend'] / trends['category_spend'].replace(0, float('nan')) * 100, 1
    ).fillna(0)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=trends['month'], y=trends['category_spend'],
                             mode='lines+markers', name='Category total',
                             line=dict(color='#607D8B', dash='dash')))
    fig.add_trace(go.Scatter(x=trends['month'], y=trends['client_spend'],
                             mode='lines+markers', name=CLIENT,
                             line=dict(color='#2E75B6', width=3)))
    fig.update_layout(title=f'Monthly spend: {CLIENT} vs {CATEGORY}',
                      xaxis_title='Month', yaxis_title='Spend (R)')
    fig.show()
elif not cat_trends.empty:
    print(f'Category trend exists but no data for {CLIENT}. Check the name.')
else:
    print(f'No monthly trends for {CATEGORY}.')

In [ ]:
if 'trends' in dir() and not trends.empty:
    fig = px.line(trends, x='month', y='client_share_pct',
                  title=f'{CLIENT} share of {CATEGORY} over time (%)',
                  color_discrete_sequence=['#E91E63'])
    fig.show()
else:
    print('No trend data to plot.')

## 5. Demographics
Who shops in this category? Age, gender, income breakdown.

In [ ]:
demo = q(f"""
    SELECT gender_label, age_group, income_group,
           SUM(customers) AS customers,
           SUM(total_spend) AS total_spend
    FROM `{PROJECT}.marts.mart_demographic_summary`
    WHERE CATEGORY_TWO = '{CATEGORY}'
    GROUP BY gender_label, age_group, income_group
""")

if not demo.empty:
    age_agg = demo.groupby('age_group').agg(customers=('customers', 'sum'),
                                             spend=('total_spend', 'sum')).reset_index()
    fig = px.bar(age_agg, x='age_group', y='customers', color='spend',
                 color_continuous_scale='Blues',
                 title=f'{CATEGORY} customers by age group')
    fig.show()
else:
    print(f'No demographic data for {CATEGORY}.')

In [ ]:
if 'demo' in dir() and not demo.empty:
    income_agg = demo.groupby('income_group').agg(customers=('customers', 'sum'),
                                                   spend=('total_spend', 'sum')).reset_index()
    fig = px.bar(income_agg, x='income_group', y='spend',
                 color='customers', color_continuous_scale='Oranges',
                 title=f'{CATEGORY} spend by income group')
    fig.show()
else:
    print('No income data to plot.')

In [ ]:
if 'demo' in dir() and not demo.empty:
    gender_agg = demo.groupby('gender_label').agg(customers=('customers', 'sum')).reset_index()
    fig = px.pie(gender_agg, values='customers', names='gender_label',
                 title=f'{CATEGORY} customers by gender')
    fig.show()
else:
    print('No gender data to plot.')

## 6. Geographic insights
Where is the spend concentrated?

In [ ]:
geo = q(f"""
    SELECT PROVINCE,
           SUM(customers) AS customers,
           SUM(total_spend) AS total_spend,
           ROUND(SAFE_DIVIDE(SUM(total_spend) * 100.0, SUM(SUM(total_spend)) OVER()), 1) AS pct
    FROM `{PROJECT}.marts.mart_geo_summary`
    WHERE CATEGORY_TWO = '{CATEGORY}'
    GROUP BY PROVINCE
    ORDER BY total_spend DESC
""")

if not geo.empty:
    fig = px.bar(geo, x='total_spend', y='PROVINCE', orientation='h',
                 color='customers', color_continuous_scale='Blues',
                 title=f'{CATEGORY} spend by province')
    fig.update_layout(yaxis=dict(autorange='reversed'))
    fig.show()

    geo[['PROVINCE', 'customers', 'total_spend', 'pct']]
else:
    print(f'No geo data for {CATEGORY}.')

## 7. ROI scenario
Quick revenue projections for the pitch.

In [ ]:
if not kpi.empty:
    k = kpi.iloc[0]
    spend_per_cust = float(k['spend_per_customer'] or 0)
    total = float(k['total_spend'] or 0)
    custs = float(k['customers'] or 0)
    
    print(f'--- ROI Scenarios for {CLIENT} ---\n')
    print(f'Current: {int(custs):,} customers, R{total:,.0f} total spend\n')
    
    if spend_per_cust > 0:
        for pct in [5, 10, 20]:
            new_customers = int(custs * pct / 100)
            new_revenue = new_customers * spend_per_cust
            print(f'  +{pct}% penetration: {new_customers:,} new customers -> +R{new_revenue:,.0f} revenue')
    
    print()
    if total > 0:
        for pct in [5, 10, 20]:
            add_spend = total * pct / 100
            print(f'  +{pct}% spend/customer: +R{add_spend:,.0f} revenue')
else:
    print(f'No KPI data for {CLIENT}. Run cell 5 first.')

## 8. Available categories and clients
Use this to find the right `CLIENT` and `CATEGORY` values.

In [ ]:
# Top categories by spend
print('Top 20 categories by total spend:\n')
cats = q(f"""
    SELECT CATEGORY_TWO, COUNT(DISTINCT DESTINATION) AS destinations,
           ROUND(SUM(total_spend), 0) AS total_spend
    FROM `{PROJECT}.marts.mart_destination_benchmarks`
    GROUP BY CATEGORY_TWO
    ORDER BY total_spend DESC
    LIMIT 20
""")
cats

In [ ]:
# Top destinations in selected category
print(f'Top 20 destinations in {CATEGORY}:\n')
dests = q(f"""
    SELECT DESTINATION, customers, total_spend, market_share_pct, spend_rank
    FROM `{PROJECT}.marts.mart_destination_benchmarks`
    WHERE CATEGORY_TWO = '{CATEGORY}'
    ORDER BY total_spend DESC
    LIMIT 20
""")
dests